# LSTM/GRU 股票预测策略

**论文参考**: Qishuo Cheng (2024), *LSTM and GRU Variants for Stock Price Prediction*

**论文核心指标**: 年化收益率 32.06%, 最大回撤 5.14%

本notebook对比三种RNN变体: Vanilla LSTM, Stacked 2-Layer LSTM, GRU，预测贵州茅台(600519)次日涨跌方向。

> **⚠️ 教学演示声明**
> 
> 本 Notebook 为量化策略算法教学样例，回测结果包含**前视偏差 (Look-ahead Bias)**：
> - 训练标签使用了未来收益率（`shift(-N)`）
> - StandardScaler 等预处理可能在全量数据上拟合
> - 模型参数选择可能基于完整样本期
> 
> **所有回测收益率仅供学习参考，不代表策略的实际可期望表现，不可直接用于实盘交易。**

## 算法原理

### LSTM (Long Short-Term Memory)

LSTM 通过三个门控机制解决传统RNN的梯度消失问题:

**遗忘门** (Forget Gate): 决定丢弃哪些历史信息
$$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

**输入门** (Input Gate): 决定存储哪些新信息
$$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
$$\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$$

**细胞状态更新**:
$$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

**输出门** (Output Gate): 决定输出哪些信息
$$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
$$h_t = o_t \odot \tanh(C_t)$$

### GRU (Gated Recurrent Unit)

GRU 将遗忘门和输入门合并为更新门，结构更简洁:

$$z_t = \sigma(W_z \cdot [h_{t-1}, x_t])$$
$$r_t = \sigma(W_r \cdot [h_{t-1}, x_t])$$
$$\tilde{h}_t = \tanh(W \cdot [r_t \odot h_{t-1}, x_t])$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

### 损失函数

二分类交叉熵:
$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N}[y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i)]$$

In [ ]:
# ===== Cell 3: LSTM门控机制动画 =====
import sys
sys.path.insert(0, '../shared')
from animation_helpers import animate_lstm_gates

fig = animate_lstm_gates(sequence_length=20, title="LSTM 门控机制 (遗忘门/输入门/输出门)")
fig.show()

In [ ]:
# ===== Cell 4: Imports & Setup =====
import sys
sys.path.insert(0, '../shared')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from data_fetcher import get_stock_daily
from factors import rsi, macd, bollinger_bands, volatility
from backtest_engine import Backtester
from visualization import plot_equity_curve, plot_drawdown, plot_metrics_table
from constants import STOCK_MOUTAI, INITIAL_CAPITAL

# 随机种子
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# ===== Cell 5: 数据获取 =====
df = get_stock_daily(STOCK_MOUTAI, start_date="20200101", end_date="20241231")
print(f"数据形状: {df.shape}")
print(f"日期范围: {df.index[0]} ~ {df.index[-1]}")
df.head()

In [ ]:
# ===== Cell 6: 特征工程 & 滑动窗口 =====
# 计算技术因子
df['close_pct'] = df['close'].pct_change()
df['volume_pct'] = df['volume'].pct_change()
df['rsi'] = rsi(df['close'], window=14)
macd_df = macd(df['close'])
df['macd_hist'] = macd_df['histogram']
bb_df = bollinger_bands(df['close'])
df['bb_pctb'] = bb_df['bb_pctb']
vol_df = volatility(df['close'], windows=[20])
df['volatility'] = vol_df['vol_20']

# 目标: 次日涨跌方向 (1=涨, 0=跌)
df['target'] = (df['close'].shift(-1) > df['close']).astype(int)

# 选择特征列
feature_cols = ['close_pct', 'volume_pct', 'rsi', 'macd_hist', 'bb_pctb', 'volatility']
df = df.dropna(subset=feature_cols + ['target'])

# 标准化特征
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# 划分训练/测试集: 2021-2023训练, 2024测试
train_mask = df.index < '2024-01-01'
test_mask = df.index >= '2024-01-01'

train_data = df[train_mask].copy()
test_data = df[test_mask].copy()

scaler.fit(train_data[feature_cols])
train_data[feature_cols] = scaler.transform(train_data[feature_cols])
test_data[feature_cols] = scaler.transform(test_data[feature_cols])

# 滑动窗口构建
WINDOW = 30

def create_sequences(data, feature_cols, target_col, window):
    X, y, dates = [], [], []
    features = data[feature_cols].values
    targets = data[target_col].values
    idx = data.index
    for i in range(window, len(data)):
        X.append(features[i-window:i])
        y.append(targets[i])
        dates.append(idx[i])
    return np.array(X), np.array(y), dates

X_train, y_train, dates_train = create_sequences(train_data, feature_cols, 'target', WINDOW)
X_test, y_test, dates_test = create_sequences(test_data, feature_cols, 'target', WINDOW)

print(f"训练集: X={X_train.shape}, y={y_train.shape}")
print(f"测试集: X={X_test.shape}, y={y_test.shape}")

# 转为PyTorch张量
train_dataset = TensorDataset(
    torch.FloatTensor(X_train), torch.FloatTensor(y_train)
)
test_dataset = TensorDataset(
    torch.FloatTensor(X_test), torch.FloatTensor(y_test)
)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# ===== Cell 7: 模型定义 & 训练 =====

class VanillaLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return torch.sigmoid(self.fc(out)).squeeze(-1)

class StackedLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=2, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return torch.sigmoid(self.fc(out)).squeeze(-1)

class GRUModel(nn.Module):
    def __init__(self, input_size=6, hidden_size=64, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        out, _ = self.gru(x)
        out = self.dropout(out[:, -1, :])
        return torch.sigmoid(self.fc(out)).squeeze(-1)


def train_model(model, train_loader, epochs=50, lr=1e-3):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    history = []
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            pred = model(X_batch)
            loss = criterion(pred, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_loader)
        history.append(avg_loss)
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
    return history


def predict_model(model, test_loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            X_batch = X_batch.to(device)
            pred = model(X_batch)
            preds.extend(pred.cpu().numpy())
    return np.array(preds)


# 训练三个模型
models = {
    'Vanilla LSTM': VanillaLSTM(),
    'Stacked LSTM': StackedLSTM(),
    'GRU': GRUModel(),
}

all_histories = {}
all_preds = {}

for name, model in models.items():
    print(f"\n--- 训练 {name} ---")
    torch.manual_seed(42)
    history = train_model(model, train_loader, epochs=50, lr=1e-3)
    all_histories[name] = history
    preds = predict_model(model, test_loader)
    all_preds[name] = preds
    acc = ((preds > 0.5).astype(int) == y_test).mean()
    print(f"  测试集准确率: {acc:.4f}")

In [ ]:
# ===== Cell 8: 预测 & 信号生成 & 回测 =====
test_prices = df.loc[dates_test, 'close']

bt = Backtester(initial_capital=INITIAL_CAPITAL)
results = {}

for name, preds in all_preds.items():
    # 信号: 预测概率 > 0.5 则买入(1), 否则空仓(0)
    signals = pd.Series((preds > 0.5).astype(int), index=dates_test)
    result = bt.run(test_prices, signals)
    results[name] = result
    print(f"\n{name} 回测结果:")
    for k, v in result['metrics'].items():
        print(f"  {k}: {v}")

In [ ]:
# ===== Cell 9: 可视化 =====

# 1. 训练损失对比
fig = go.Figure()
colors = ['#2196F3', '#F44336', '#4CAF50']
for i, (name, hist) in enumerate(all_histories.items()):
    fig.add_trace(go.Scatter(y=hist, mode='lines', name=name,
                             line=dict(color=colors[i], width=2)))
fig.update_layout(title='训练损失对比', xaxis_title='Epoch', yaxis_title='Loss',
                  height=400, width=800)
fig.show()

# 2. 预测概率 vs 实际涨跌
best_name = max(results, key=lambda n: float(results[n]['metrics']['年化收益率'].strip('%')))
best_preds = all_preds[best_name]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=[f'{best_name} 预测概率 vs 实际', '收盘价走势'])
fig.add_trace(go.Scatter(x=dates_test, y=best_preds, mode='lines',
                         name='预测概率', line=dict(color='#2196F3')), row=1, col=1)
fig.add_trace(go.Scatter(x=dates_test, y=y_test, mode='markers',
                         name='实际涨跌', marker=dict(size=3, color='gray', opacity=0.4)), row=1, col=1)
fig.add_hline(y=0.5, line_dash='dash', line_color='red', row=1, col=1)
fig.add_trace(go.Scatter(x=dates_test, y=test_prices.values, mode='lines',
                         name='收盘价', line=dict(color='#333')), row=2, col=1)
fig.update_layout(height=600, width=1000, title=f'{best_name} 预测结果')
fig.show()

# 3. 权益曲线对比
fig = go.Figure()
for i, (name, res) in enumerate(results.items()):
    eq = res['equity_curve']
    fig.add_trace(go.Scatter(x=eq.index, y=eq.values / eq.iloc[0],
                             mode='lines', name=name, line=dict(color=colors[i], width=2)))
# 基准: 买入持有
bh = test_prices / test_prices.iloc[0]
fig.add_trace(go.Scatter(x=bh.index, y=bh.values, mode='lines',
                         name='买入持有', line=dict(color='gray', dash='dash')))
fig.update_layout(title='三种模型权益曲线对比', xaxis_title='日期',
                  yaxis_title='累计净值', height=500, width=1000)
fig.show()

# 4. 绩效指标表
best_result = results[best_name]
plot_metrics_table(best_result['metrics'], title=f"{best_name} 绩效指标")

# 5. 回撤图
plot_drawdown(best_result['equity_curve'], title=f"{best_name} 回撤")

## 结果讨论

### 模型对比
- **Vanilla LSTM**: 单层LSTM作为基准，参数量最少，训练最快
- **Stacked LSTM**: 2层堆叠增强了特征提取能力，但可能过拟合
- **GRU**: 参数量介于两者之间，更新门和重置门的简化设计有时表现更优

### 关键发现
1. RNN类模型能捕捉短期动量特征，30天窗口适合捕获月度趋势
2. 市场状态(趋势/震荡)对模型表现有显著影响，震荡期准确率下降
3. 特征工程(RSI/MACD/布林带)对模型性能贡献显著

### 局限性
- 单股票回测存在生存者偏差
- 未考虑流动性约束和冲击成本
- 实际应用需结合仓位管理和风控规则

### 参考
- Cheng, Q. (2024). LSTM and GRU variants for stock price prediction.